# Yelp Dataset — EDA & Feature Engineering
### Medallion Architecture: Bronze → Silver → Gold
| Layer | Description | Storage |
|-------|-------------|----------|
| **Bronze** | Raw Parquet files | `Dataset/parquet/` |
| **Silver** | Cleaned, typed, validated datasets | `Dataset/results/silver/` |
| **Gold** | Joined, feature-engineered, ML-ready dataset | `Dataset/results/gold/` |

**Datasets:** Business · Review · User · Tip · Checkin

## 1 — Install Dependencies

In [1]:
!pip install pyspark pyarrow matplotlib seaborn wordcloud nltk -q

## 2 — Imports & Configuration

In [2]:
import os, re, warnings
from pathlib import Path

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, FloatType, StringType
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
sns.set_palette('husl')
STOP_WORDS = set(stopwords.words('english'))
print('Libraries loaded.')

Libraries loaded.


## 3 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4 — Paths & Output Directories

In [4]:
PARQUET_DIR = '/content/drive/MyDrive/Dataset/parquet'
RESULTS_DIR = '/content/drive/MyDrive/Dataset/results'
SILVER_DIR  = f'{RESULTS_DIR}/silver'
GOLD_DIR    = f'{RESULTS_DIR}/gold'
EDA_DIR     = f'{RESULTS_DIR}/EDA'

for d in [RESULTS_DIR, SILVER_DIR, GOLD_DIR, EDA_DIR]:
    os.makedirs(d, exist_ok=True)

def save_fig(name):
    path = os.path.join(EDA_DIR, f'{name}.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Saved → {path}')

print('Output directories ready:')
for d in [SILVER_DIR, GOLD_DIR, EDA_DIR]:
    print(f'  {d}')

Output directories ready:
  /content/drive/MyDrive/Dataset/results/silver
  /content/drive/MyDrive/Dataset/results/gold
  /content/drive/MyDrive/Dataset/results/EDA


In [13]:
import pyarrow as pa
import pyarrow.parquet as pq

def fix_parquet_nanoseconds(file_path):
    """Re-write a parquet file replacing TIMESTAMP(NANOS) with TIMESTAMP(MICROS)."""
    reader = pq.ParquetFile(file_path)

    # Check if any column is nanosecond timestamp
    schema = reader.schema_arrow
    needs_fix = any(
        pa.types.is_timestamp(field.type) and field.type.unit == 'ns'
        for field in schema
    )
    if not needs_fix:
        print(f'  OK (no nanosecond columns): {Path(file_path).name}')
        return

    print(f'  Fixing nanosecond timestamps: {Path(file_path).name} ...')
    tmp_path = file_path + '.tmp'
    writer = None

    for batch in reader.iter_batches(batch_size=100_000):
        table = pa.Table.from_batches([batch])
        new_cols = {}
        for name in table.schema.names:
            col = table.column(name)
            if pa.types.is_timestamp(col.type) and col.type.unit == 'ns':
                col = col.cast(pa.timestamp('us'))
            new_cols[name] = col
        fixed = pa.table(new_cols)
        if writer is None:
            writer = pq.ParquetWriter(tmp_path, fixed.schema, compression='snappy')
        writer.write_table(fixed)

    if writer:
        writer.close()

    # Replace original file
    os.replace(tmp_path, file_path)
    print(f'    Done.')

print('Checking parquet files for nanosecond timestamps ...\n')
parquet_files = list(Path(PARQUET_DIR).glob('*.parquet'))
for pf in parquet_files:
    fix_parquet_nanoseconds(str(pf))

print('\nAll files OK — safe to load with Spark.')

Checking parquet files for nanosecond timestamps ...

  OK (no nanosecond columns): yelp_academic_dataset_user.parquet
  Fixing nanosecond timestamps: yelp_academic_dataset_tip.parquet ...
    Done.
  OK (no nanosecond columns): yelp_academic_dataset_business.parquet
  OK (no nanosecond columns): yelp_academic_dataset_checkin.parquet
  Fixing nanosecond timestamps: yelp_academic_dataset_review.parquet ...
    Done.

All files OK — safe to load with Spark.


## 5 — Start Spark Session

In [14]:
spark = (
    SparkSession.builder
    .appName('YelpEDA_FeatureEngineering')
    .config('spark.driver.memory', '8g')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.shuffle.partitions', '50')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')

Spark 4.0.2 ready.


## 6 — Load Bronze Data

In [15]:
business_df = spark.read.parquet(f'{PARQUET_DIR}/yelp_academic_dataset_business.parquet')
review_df   = spark.read.parquet(f'{PARQUET_DIR}/yelp_academic_dataset_review.parquet')
user_df     = spark.read.parquet(f'{PARQUET_DIR}/yelp_academic_dataset_user.parquet')
tip_df      = spark.read.parquet(f'{PARQUET_DIR}/yelp_academic_dataset_tip.parquet')
checkin_df  = spark.read.parquet(f'{PARQUET_DIR}/yelp_academic_dataset_checkin.parquet')

# Cache smaller tables used repeatedly
business_df.cache()
user_df.cache()

datasets = {
    'Business' : business_df,
    'Review'   : review_df,
    'User'     : user_df,
    'Tip'      : tip_df,
    'Checkin'  : checkin_df,
}

print(f'{'Dataset':<12} {'Rows':>12}  {'Columns':>8}')
print('-' * 36)
for name, df in datasets.items():
    print(f'{name:<12} {df.count():>12,}  {len(df.columns):>8}')

Dataset              Rows   Columns
------------------------------------
Business          150,346        14
Review          6,990,280         9
User            1,987,897        22
Tip               908,915         5
Checkin           131,930         2


---
# EDA — Exploratory Data Analysis

## 7 — EDA: Business Dataset

In [16]:
# Schema & Null Analysis
print('=== BUSINESS SCHEMA ===')
business_df.printSchema()

total = business_df.count()
null_expr = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in business_df.columns]
null_pd = business_df.select(null_expr).toPandas().T
null_pd.columns = ['null_count']
null_pd['null_pct'] = (null_pd['null_count'] / total * 100).round(2)
print('\n=== NULL ANALYSIS ===')
print(null_pd[null_pd['null_count'] > 0].sort_values('null_pct', ascending=False).to_string())

=== BUSINESS SCHEMA ===
root
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- stars: double (nullable = true)
 |-- review_count: long (nullable = true)
 |-- is_open: long (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |

In [18]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Business Dataset — EDA', fontsize=15, fontweight='bold')

# 1. Star rating distribution
stars_pd = business_df.groupBy('stars').count().orderBy('stars').toPandas()
axes[0,0].bar(stars_pd['stars'].astype(str), stars_pd['count'], color=sns.color_palette('RdYlGn', len(stars_pd)))
axes[0,0].set_title('Star Rating Distribution')
axes[0,0].set_xlabel('Stars'); axes[0,0].set_ylabel('Number of Businesses')
for bar, val in zip(axes[0,0].patches, stars_pd['count']):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                   f'{val:,}', ha='center', fontsize=8)

# 2. Open vs Closed
open_pd = business_df.groupBy('is_open').count().toPandas()
labels  = ['Closed', 'Open']
colors  = ['#e74c3c', '#2ecc71']
axes[0,1].pie(open_pd.sort_values('is_open')['count'], labels=labels, colors=colors,
              autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0,1].set_title('Open vs Closed Businesses')

# 3. Top 10 states
state_pd = business_df.groupBy('state').count().orderBy(F.desc('count')).limit(10).toPandas()
axes[1,0].barh(state_pd['state'][::-1], state_pd['count'][::-1], color='steelblue')
axes[1,0].set_title('Top 10 States by Business Count')
axes[1,0].set_xlabel('Number of Businesses')

# 4. Average stars by state (top 10)
avg_state = business_df.groupBy('state').agg(
    F.avg('stars').alias('avg_stars'), F.count('*').alias('cnt')
).filter(F.col('cnt') > 500).orderBy(F.desc('avg_stars')).limit(10).toPandas()
bars = axes[1,1].bar(avg_state['state'], avg_state['avg_stars'],
                      color=sns.color_palette('coolwarm', len(avg_state)))
axes[1,1].set_title('Avg Star Rating by State (min 500 businesses)')
axes[1,1].set_xlabel('State'); axes[1,1].set_ylabel('Average Stars')
axes[1,1].set_ylim(3.0, 4.5)

plt.show()
save_fig('01_business_overview')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/01_business_overview.png


In [19]:
# Top 20 categories
cat_df = business_df.filter(F.col('categories').isNotNull()) \
    .select(F.explode(F.split('categories', ', ')).alias('category'))
top_cats = cat_df.groupBy('category').count().orderBy(F.desc('count')).limit(20).toPandas()

plt.figure(figsize=(12, 7))
plt.barh(top_cats['category'][::-1], top_cats['count'][::-1],
         color=sns.color_palette('viridis', 20))
plt.title('Top 20 Business Categories', fontsize=14, fontweight='bold')
plt.xlabel('Number of Businesses')
for i, (v, c) in enumerate(zip(top_cats['count'][::-1], top_cats['category'][::-1])):
    plt.text(v + 50, i, f'{v:,}', va='center', fontsize=8)
plt.tight_layout()
save_fig('02_business_top_categories')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/02_business_top_categories.png


## 8 — EDA: Review Dataset

In [20]:
print('=== REVIEW SCHEMA ===')
review_df.printSchema()

total = review_df.count()
null_expr = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in review_df.columns]
null_pd = review_df.select(null_expr).toPandas().T
null_pd.columns = ['null_count']
null_pd['null_pct'] = (null_pd['null_count'] / total * 100).round(2)
print(f'\nTotal rows: {total:,}')
print('\n=== NULL ANALYSIS ===')
display(null_pd.sort_values('null_pct', ascending=False))

=== REVIEW SCHEMA ===
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: long (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- text: string (nullable = true)
 |-- date: timestamp_ntz (nullable = true)


Total rows: 6,990,280

=== NULL ANALYSIS ===


,null_count,null_pct
review_id,0,0.0
user_id,0,0.0
business_id,0,0.0
stars,0,0.0
useful,0,0.0
funny,0,0.0
cool,0,0.0
text,0,0.0
date,0,0.0


In [21]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Review Dataset — EDA', fontsize=15, fontweight='bold')

# 1. Star distribution
rev_stars = review_df.groupBy('stars').count().orderBy('stars').toPandas()
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
bars = axes[0,0].bar(rev_stars['stars'].astype(str), rev_stars['count'], color=colors)
axes[0,0].set_title('Review Star Distribution'); axes[0,0].set_xlabel('Stars')
axes[0,0].set_ylabel('Number of Reviews')
for bar, val in zip(bars, rev_stars['count']):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                   f'{val/1e6:.2f}M', ha='center', fontsize=9)

# 2. Reviews per year
reviews_yr = review_df.withColumn('year', F.year('date')) \
    .groupBy('year').count().orderBy('year').toPandas()
reviews_yr = reviews_yr[reviews_yr['year'].between(2005, 2022)]
axes[0,1].plot(reviews_yr['year'], reviews_yr['count'], marker='o', linewidth=2, color='steelblue')
axes[0,1].fill_between(reviews_yr['year'], reviews_yr['count'], alpha=0.2, color='steelblue')
axes[0,1].set_title('Reviews per Year'); axes[0,1].set_xlabel('Year')
axes[0,1].set_ylabel('Number of Reviews')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Review text length distribution (sample)
rev_len = review_df.withColumn('text_len', F.length('text')) \
    .select('text_len').sample(fraction=0.05, seed=42).toPandas()
rev_len_clipped = rev_len[rev_len['text_len'] < 3000]
axes[1,0].hist(rev_len_clipped['text_len'], bins=60, color='mediumorchid', edgecolor='white', linewidth=0.5)
axes[1,0].set_title('Review Text Length Distribution (5% sample)')
axes[1,0].set_xlabel('Character Count'); axes[1,0].set_ylabel('Frequency')

# 4. Votes analysis
votes = review_df.agg(
    F.avg('useful').alias('avg_useful'),
    F.avg('funny').alias('avg_funny'),
    F.avg('cool').alias('avg_cool'),
    F.sum(F.when(F.col('useful') > 0, 1).otherwise(0)).alias('has_useful'),
    F.sum(F.when(F.col('funny') > 0, 1).otherwise(0)).alias('has_funny'),
    F.sum(F.when(F.col('cool') > 0, 1).otherwise(0)).alias('has_cool'),
).toPandas().T
avg_votes = {'Useful': float(votes.loc['avg_useful']), 'Funny': float(votes.loc['avg_funny']), 'Cool': float(votes.loc['avg_cool'])}
axes[1,1].bar(avg_votes.keys(), avg_votes.values(), color=['#3498db','#e74c3c','#2ecc71'])
axes[1,1].set_title('Average Vote Counts per Review')
axes[1,1].set_ylabel('Average Count')

plt.tight_layout()
save_fig('03_review_overview')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/03_review_overview.png


In [22]:
# Reviews by month & day of week
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Review Temporal Patterns', fontsize=14, fontweight='bold')

monthly = review_df.withColumn('month', F.month('date')) \
    .groupBy('month').count().orderBy('month').toPandas()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[0].bar(month_names, monthly['count'], color=sns.color_palette('tab10', 12))
axes[0].set_title('Reviews by Month'); axes[0].set_ylabel('Count')

daily = review_df.withColumn('dow', F.dayofweek('date')) \
    .groupBy('dow').count().orderBy('dow').toPandas()
dow_names = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']
axes[1].bar(dow_names, daily['count'], color=sns.color_palette('tab10', 7))
axes[1].set_title('Reviews by Day of Week'); axes[1].set_ylabel('Count')

plt.tight_layout()
save_fig('04_review_temporal')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/04_review_temporal.png


In [23]:
# Word Cloud from review text (5k sample)
sample_text = review_df.select('text').sample(fraction=0.001, seed=42).toPandas()
corpus = ' '.join(sample_text['text'].dropna().tolist())
words  = ' '.join([w.lower() for w in corpus.split() if w.lower() not in STOP_WORDS and len(w) > 2])

wc = WordCloud(width=1200, height=500, background_color='white',
               colormap='viridis', max_words=200).generate(words)
plt.figure(figsize=(14, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Reviews (stopwords removed)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('05_review_wordcloud')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/05_review_wordcloud.png


## 9 — EDA: User Dataset

In [24]:
print('=== USER SCHEMA ===')
user_df.printSchema()

total = user_df.count()
null_expr = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in user_df.columns]
null_pd = user_df.select(null_expr).toPandas().T
null_pd.columns = ['null_count']
null_pd['null_pct'] = (null_pd['null_count'] / total * 100).round(2)
print(f'\nTotal rows: {total:,}')
print('\n=== NULL ANALYSIS ===')
display(null_pd.sort_values('null_pct', ascending=False))

=== USER SCHEMA ===
root
 |-- user_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- yelping_since: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- elite: string (nullable = true)
 |-- friends: string (nullable = true)
 |-- fans: long (nullable = true)
 |-- average_stars: double (nullable = true)
 |-- compliment_hot: long (nullable = true)
 |-- compliment_more: long (nullable = true)
 |-- compliment_profile: long (nullable = true)
 |-- compliment_cute: long (nullable = true)
 |-- compliment_list: long (nullable = true)
 |-- compliment_note: long (nullable = true)
 |-- compliment_plain: long (nullable = true)
 |-- compliment_cool: long (nullable = true)
 |-- compliment_funny: long (nullable = true)
 |-- compliment_writer: long (nullable = true)
 |-- compliment_photos: long (nullable = true)


Total rows: 1,987,897

=== NULL ANALYSIS ===


,null_count,null_pct
user_id,0,0.0
name,0,0.0
review_count,0,0.0
yelping_since,0,0.0
useful,0,0.0
funny,0,0.0
cool,0,0.0
elite,0,0.0
friends,0,0.0
fans,0,0.0


In [25]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('User Dataset — EDA', fontsize=15, fontweight='bold')

# 1. Review count distribution (log scale)
rc_sample = user_df.select('review_count').sample(fraction=0.1, seed=42).toPandas()
axes[0,0].hist(np.log1p(rc_sample['review_count']), bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('User Review Count Distribution (log scale)')
axes[0,0].set_xlabel('log(1 + review_count)'); axes[0,0].set_ylabel('Frequency')

# 2. Elite vs Non-elite
elite_pd = user_df.withColumn('is_elite', F.when(
    (F.col('elite').isNull()) | (F.col('elite') == ''), F.lit('Non-Elite')
).otherwise(F.lit('Elite'))) \
    .groupBy('is_elite').count().toPandas()
axes[0,1].pie(elite_pd['count'], labels=elite_pd['is_elite'],
              colors=['#f39c12','#95a5a6'], autopct='%1.1f%%',
              startangle=90, textprops={'fontsize': 12})
axes[0,1].set_title('Elite vs Non-Elite Users')

# 3. Average stars given by users
avg_s = user_df.select('average_stars').sample(fraction=0.1, seed=42).toPandas()
axes[1,0].hist(avg_s['average_stars'], bins=40, color='mediumorchid', edgecolor='white')
axes[1,0].set_title('Distribution of User Average Stars Given')
axes[1,0].set_xlabel('Average Stars'); axes[1,0].set_ylabel('Frequency')

# 4. Fans distribution (log scale)
fans = user_df.select('fans').filter(F.col('fans') > 0).sample(fraction=0.1, seed=42).toPandas()
axes[1,1].hist(np.log1p(fans['fans']), bins=40, color='tomato', edgecolor='white')
axes[1,1].set_title('Fans Distribution (log scale, fans > 0)')
axes[1,1].set_xlabel('log(1 + fans)'); axes[1,1].set_ylabel('Frequency')

plt.tight_layout()
save_fig('06_user_overview')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/06_user_overview.png


In [26]:
# User yelping since — membership trend
user_yr = user_df.withColumn('join_year', F.year(F.to_timestamp('yelping_since'))) \
    .groupBy('join_year').count().orderBy('join_year').toPandas()
user_yr = user_yr[user_yr['join_year'].between(2004, 2022)]

plt.figure(figsize=(12, 4))
plt.fill_between(user_yr['join_year'], user_yr['count'], alpha=0.4, color='teal')
plt.plot(user_yr['join_year'], user_yr['count'], marker='o', color='teal', linewidth=2)
plt.title('New User Registrations per Year', fontsize=14, fontweight='bold')
plt.xlabel('Year'); plt.ylabel('New Users')
plt.xticks(user_yr['join_year'], rotation=45)
plt.tight_layout()
save_fig('07_user_growth')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/07_user_growth.png


## 10 — EDA: Tip & Checkin Datasets

In [27]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Tip Dataset — EDA', fontsize=14, fontweight='bold')

# Tips per year
tip_yr = tip_df.withColumn('year', F.year('date')) \
    .groupBy('year').count().orderBy('year').toPandas()
tip_yr = tip_yr[tip_yr['year'].between(2009, 2022)]
axes[0].bar(tip_yr['year'].astype(str), tip_yr['count'], color='coral')
axes[0].set_title('Tips per Year'); axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Tips')
axes[0].tick_params(axis='x', rotation=45)

# Compliment count distribution
comp = tip_df.select('compliment_count').toPandas()
axes[1].hist(np.log1p(comp['compliment_count']), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Tip Compliment Count (log scale)')
axes[1].set_xlabel('log(1 + compliment_count)'); axes[1].set_ylabel('Frequency')

plt.tight_layout()
save_fig('08_tip_overview')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/08_tip_overview.png


In [28]:
# Checkin — parse comma-separated dates, analyze by hour and day
checkin_exploded = checkin_df.select(
    F.explode(F.split('date', ', ')).alias('checkin_date')
).withColumn('checkin_ts', F.to_timestamp('checkin_date', 'yyyy-MM-dd HH:mm:ss')) \
 .withColumn('hour', F.hour('checkin_ts')) \
 .withColumn('dow',  F.dayofweek('checkin_ts'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Checkin Dataset — EDA', fontsize=14, fontweight='bold')

hourly = checkin_exploded.groupBy('hour').count().orderBy('hour').toPandas()
axes[0].bar(hourly['hour'], hourly['count'], color='slateblue')
axes[0].set_title('Checkins by Hour of Day')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Checkin Count')
axes[0].set_xticks(range(0, 24))

dow_ci = checkin_exploded.groupBy('dow').count().orderBy('dow').toPandas()
dow_names = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']
axes[1].bar(dow_names, dow_ci['count'], color='slateblue')
axes[1].set_title('Checkins by Day of Week')
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Checkin Count')

plt.tight_layout()
save_fig('09_checkin_patterns')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/09_checkin_patterns.png


---
# Silver Layer — Data Cleaning

## 11 — Silver: Clean Business

In [29]:
silver_business = business_df \
    .filter(F.col('business_id').isNotNull()) \
    .withColumn('latitude',  F.col('latitude').cast(FloatType())) \
    .withColumn('longitude', F.col('longitude').cast(FloatType())) \
    .withColumn('stars',       F.col('stars').cast(FloatType())) \
    .withColumn('review_count', F.col('review_count').cast(IntegerType())) \
    .withColumn('is_open',     F.col('is_open').cast(IntegerType())) \
    .withColumn('city',        F.trim(F.lower(F.col('city')))) \
    .withColumn('state',       F.trim(F.upper(F.col('state')))) \
    .withColumn('postal_code', F.trim(F.col('postal_code'))) \
    .withColumn('category_list', F.split(F.col('categories'), ', ')) \
    .withColumn('category_count', F.size('category_list')) \
    .withColumn('category_count',
                F.when(F.col('category_count') < 0, 0).otherwise(F.col('category_count'))) \
    .fillna({'stars': 0.0, 'review_count': 0, 'is_open': 0}) \
    .drop('attributes', 'hours')  # nested structs — drop for tabular layer

print(f'Silver Business: {silver_business.count():,} rows x {len(silver_business.columns)} cols')
silver_business.printSchema()

Silver Business: 150,346 rows x 14 cols
root
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- latitude: float (nullable = true)
 |-- longitude: float (nullable = true)
 |-- stars: float (nullable = false)
 |-- review_count: integer (nullable = false)
 |-- is_open: integer (nullable = false)
 |-- categories: string (nullable = true)
 |-- category_list: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- category_count: integer (nullable = true)



## 12 — Silver: Clean Review

In [30]:
silver_review = review_df \
    .filter(F.col('review_id').isNotNull() & F.col('text').isNotNull()) \
    .withColumn('stars',   F.col('stars').cast(IntegerType())) \
    .withColumn('useful',  F.col('useful').cast(IntegerType())) \
    .withColumn('funny',   F.col('funny').cast(IntegerType())) \
    .withColumn('cool',    F.col('cool').cast(IntegerType())) \
    .withColumn('date',    F.to_timestamp('date')) \
    .withColumn('text',    F.trim(F.col('text'))) \
    .filter(F.col('stars').between(1, 5)) \
    .fillna({'useful': 0, 'funny': 0, 'cool': 0})

print(f'Silver Review: {silver_review.count():,} rows x {len(silver_review.columns)} cols')

Silver Review: 6,990,280 rows x 9 cols


## 13 — Silver: Clean User

In [31]:
silver_user = user_df \
    .filter(F.col('user_id').isNotNull()) \
    .withColumn('yelping_since', F.to_timestamp('yelping_since')) \
    .withColumn('review_count',  F.col('review_count').cast(IntegerType())) \
    .withColumn('useful',        F.col('useful').cast(IntegerType())) \
    .withColumn('funny',         F.col('funny').cast(IntegerType())) \
    .withColumn('cool',          F.col('cool').cast(IntegerType())) \
    .withColumn('fans',          F.col('fans').cast(IntegerType())) \
    .withColumn('average_stars', F.col('average_stars').cast(FloatType())) \
    .withColumn('is_elite',
                F.when((F.col('elite').isNull()) | (F.col('elite') == ''), 0).otherwise(1)) \
    .withColumn('elite_years_count',
                F.when((F.col('elite').isNull()) | (F.col('elite') == ''), 0)
                 .otherwise(F.size(F.split(F.col('elite'), ',')))) \
    .withColumn('tenure_days',
                F.datediff(F.current_date(), F.to_date('yelping_since'))) \
    .fillna({'review_count': 0, 'useful': 0, 'funny': 0, 'cool': 0, 'fans': 0, 'average_stars': 0.0})

print(f'Silver User: {silver_user.count():,} rows x {len(silver_user.columns)} cols')

Silver User: 1,987,897 rows x 25 cols


## 14 — Silver: Clean Tip & Checkin

In [32]:
silver_tip = tip_df \
    .filter(F.col('user_id').isNotNull() & F.col('text').isNotNull()) \
    .withColumn('date', F.to_timestamp('date')) \
    .withColumn('compliment_count', F.col('compliment_count').cast(IntegerType())) \
    .withColumn('text', F.trim(F.col('text'))) \
    .fillna({'compliment_count': 0})

silver_checkin = checkin_df \
    .filter(F.col('business_id').isNotNull()) \
    .withColumn('checkin_count', F.size(F.split(F.col('date'), ', ')))

print(f'Silver Tip     : {silver_tip.count():,} rows')
print(f'Silver Checkin : {silver_checkin.count():,} rows')

Silver Tip     : 908,915 rows
Silver Checkin : 131,930 rows


## 15 — Save Silver Layer

In [33]:
silver_datasets = {
    'business' : silver_business,
    'review'   : silver_review,
    'user'     : silver_user,
    'tip'      : silver_tip,
    'checkin'  : silver_checkin,
}

for name, df in silver_datasets.items():
    path = f'{SILVER_DIR}/{name}.parquet'
    df.write.mode('overwrite').parquet(path)
    print(f'  Saved silver/{name}.parquet')

print('\nSilver layer saved.')

  Saved silver/business.parquet
  Saved silver/review.parquet
  Saved silver/user.parquet
  Saved silver/tip.parquet
  Saved silver/checkin.parquet

Silver layer saved.


---
# Gold Layer — Feature Engineering

## 16 — Join: Review + Business + User

In [34]:
# Select only needed columns for the join to keep memory manageable
biz_sel = silver_business.select(
    'business_id',
    F.col('stars').alias('biz_stars'),
    F.col('review_count').alias('biz_review_count'),
    F.col('is_open').alias('biz_is_open'),
    F.col('category_count').alias('biz_category_count'),
    F.col('state').alias('biz_state'),
    F.col('city').alias('biz_city'),
    F.col('latitude').alias('biz_latitude'),
    F.col('longitude').alias('biz_longitude'),
)

usr_sel = silver_user.select(
    'user_id',
    F.col('review_count').alias('user_review_count'),
    F.col('average_stars').alias('user_avg_stars'),
    F.col('useful').alias('user_useful_votes'),
    F.col('funny').alias('user_funny_votes'),
    F.col('cool').alias('user_cool_votes'),
    F.col('fans').alias('user_fans'),
    F.col('is_elite').alias('user_is_elite'),
    F.col('elite_years_count').alias('user_elite_years'),
    F.col('tenure_days').alias('user_tenure_days'),
)

gold_df = silver_review \
    .join(biz_sel, on='business_id', how='left') \
    .join(usr_sel, on='user_id',     how='left')

print(f'Joined: {gold_df.count():,} rows x {len(gold_df.columns)} cols')
gold_df.printSchema()

Joined: 6,990,280 rows x 26 cols
root
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: integer (nullable = true)
 |-- useful: integer (nullable = false)
 |-- funny: integer (nullable = false)
 |-- cool: integer (nullable = false)
 |-- text: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- biz_stars: float (nullable = true)
 |-- biz_review_count: integer (nullable = true)
 |-- biz_is_open: integer (nullable = true)
 |-- biz_category_count: integer (nullable = true)
 |-- biz_state: string (nullable = true)
 |-- biz_city: string (nullable = true)
 |-- biz_latitude: float (nullable = true)
 |-- biz_longitude: float (nullable = true)
 |-- user_review_count: integer (nullable = true)
 |-- user_avg_stars: float (nullable = true)
 |-- user_useful_votes: integer (nullable = true)
 |-- user_funny_votes: integer (nullable = true)
 |-- user_cool_votes: integer (nullable = true)
 |-- user_fans:

## 17 — Text Features

In [35]:
gold_df = gold_df \
    .withColumn('char_count',       F.length('text')) \
    .withColumn('word_count',       F.size(F.split(F.trim('text'), r'\s+'))) \
    .withColumn('avg_word_length',
                (F.length(F.regexp_replace('text', r'\s+', '')) / F.col('word_count')).cast(FloatType())) \
    .withColumn('exclamation_count', F.size(F.split('text', r'!')) - 1) \
    .withColumn('question_count',    F.size(F.split('text', r'\?')) - 1) \
    .withColumn('uppercase_count',
                F.length(F.regexp_replace('text', r'[^A-Z]', ''))) \
    .withColumn('uppercase_ratio',
                (F.col('uppercase_count') / F.col('char_count')).cast(FloatType())) \
    .withColumn('sentence_count', F.size(F.split('text', r'[.!?]+')) - 1)

print('Text features added.')
gold_df.select('char_count','word_count','avg_word_length',
               'exclamation_count','question_count','uppercase_ratio','sentence_count').show(5)

Text features added.
+----------+----------+---------------+-----------------+--------------+---------------+--------------+
|char_count|word_count|avg_word_length|exclamation_count|question_count|uppercase_ratio|sentence_count|
+----------+----------+---------------+-----------------+--------------+---------------+--------------+
|       558|       110|       4.081818|                1|             0|    0.044802867|             6|
|       372|        67|       4.567164|                0|             0|      0.1155914|             5|
|       369|        70|       4.285714|                1|             0|    0.048780486|             4|
|       511|       102|       4.019608|                0|             0|     0.06066536|             7|
|       322|        63|       4.126984|                4|             0|     0.03416149|             2|
+----------+----------+---------------+-----------------+--------------+---------------+--------------+
only showing top 5 rows


## 18 — Temporal Features

In [36]:
gold_df = gold_df \
    .withColumn('review_year',  F.year('date').cast(IntegerType())) \
    .withColumn('review_month', F.month('date').cast(IntegerType())) \
    .withColumn('review_dow',   F.dayofweek('date').cast(IntegerType())) \
    .withColumn('review_hour',  F.hour('date').cast(IntegerType())) \
    .withColumn('is_weekend',
                F.when(F.dayofweek('date').isin([1, 7]), 1).otherwise(0))

print('Temporal features added.')
gold_df.select('date','review_year','review_month','review_dow','review_hour','is_weekend').show(5)

Temporal features added.
+-------------------+-----------+------------+----------+-----------+----------+
|               date|review_year|review_month|review_dow|review_hour|is_weekend|
+-------------------+-----------+------------+----------+-----------+----------+
|2018-10-03 21:44:03|       2018|          10|         4|         21|         0|
|2017-10-04 01:55:12|       2017|          10|         4|          1|         0|
|2015-03-11 05:07:09|       2015|           3|         4|          5|         0|
|2018-11-27 02:18:43|       2018|          11|         3|          2|         0|
|2011-04-05 14:07:42|       2011|           4|         3|         14|         0|
+-------------------+-----------+------------+----------+-----------+----------+
only showing top 5 rows


## 19 — Interaction & Aggregate Features

In [37]:
# Star deviation: how this review differs from user's usual rating
gold_df = gold_df \
    .withColumn('user_star_deviation',
                (F.col('stars') - F.col('user_avg_stars')).cast(FloatType())) \
    .withColumn('biz_star_deviation',
                (F.col('stars') - F.col('biz_stars')).cast(FloatType())) \
    .withColumn('total_votes',
                (F.col('useful') + F.col('funny') + F.col('cool')).cast(IntegerType())) \
    .withColumn('user_engagement_score',
                (F.log1p('user_review_count') + F.log1p('user_fans') +
                 F.log1p('user_useful_votes')).cast(FloatType()))

print('Interaction features added.')
gold_df.select('user_star_deviation','biz_star_deviation',
               'total_votes','user_engagement_score').show(5)

Interaction features added.
+-------------------+------------------+-----------+---------------------+
|user_star_deviation|biz_star_deviation|total_votes|user_engagement_score|
+-------------------+------------------+-----------+---------------------+
|          1.8599999|               0.0|          0|            2.9957323|
|         0.44000006|               0.5|          0|            4.0253515|
|         -1.0699999|              -2.0|          1|            5.3936276|
|        -0.43000007|              -2.0|          3|             4.787492|
|          0.9299998|               0.5|          3|            17.948706|
+-------------------+------------------+-----------+---------------------+
only showing top 5 rows


## 20 — Sentiment Label (Target Variable)

In [38]:
# 3-class sentiment label
# 0 = Negative  (1–2 stars)
# 1 = Neutral   (3 stars)
# 2 = Positive  (4–5 stars)
gold_df = gold_df.withColumn('sentiment_label',
    F.when(F.col('stars').isin([1, 2]), 0)
     .when(F.col('stars') == 3, 1)
     .otherwise(2)
)

# Binary label (exclude neutral)
gold_df = gold_df.withColumn('sentiment_binary',
    F.when(F.col('stars').isin([1, 2]), 0)
     .when(F.col('stars').isin([4, 5]), 1)
     .otherwise(None)
)

label_dist = gold_df.groupBy('sentiment_label').count().orderBy('sentiment_label').toPandas()
print('3-class label distribution:')
label_dist['label_name'] = label_dist['sentiment_label'].map({0:'Negative', 1:'Neutral', 2:'Positive'})
label_dist['pct'] = (label_dist['count'] / label_dist['count'].sum() * 100).round(2)
display(label_dist[['label_name','count','pct']])

3-class label distribution:


,label_name,count,pct
0,Negative,1613801,23.09
1,Neutral,691934,9.90
2,Positive,4684545,67.02


## 21 — Gold Layer: Final Schema & Save

In [39]:
# Select and order final columns
gold_final = gold_df.select(
    # IDs
    'review_id', 'user_id', 'business_id',
    # Target labels
    'stars', 'sentiment_label', 'sentiment_binary',
    # Review content
    'text',
    # Text features
    'char_count', 'word_count', 'avg_word_length',
    'exclamation_count', 'question_count', 'uppercase_count',
    'uppercase_ratio', 'sentence_count',
    # Review vote features
    'useful', 'funny', 'cool', 'total_votes',
    # Temporal features
    'date', 'review_year', 'review_month', 'review_dow', 'review_hour', 'is_weekend',
    # User features
    'user_review_count', 'user_avg_stars', 'user_useful_votes',
    'user_funny_votes', 'user_cool_votes', 'user_fans',
    'user_is_elite', 'user_elite_years', 'user_tenure_days',
    'user_engagement_score',
    # Business features
    'biz_stars', 'biz_review_count', 'biz_is_open',
    'biz_category_count', 'biz_state', 'biz_city',
    'biz_latitude', 'biz_longitude',
    # Interaction features
    'user_star_deviation', 'biz_star_deviation',
)

gold_count = gold_final.count()
print(f'Gold layer: {gold_count:,} rows x {len(gold_final.columns)} features')
gold_final.printSchema()

Gold layer: 6,990,280 rows x 45 features
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: integer (nullable = true)
 |-- sentiment_label: integer (nullable = false)
 |-- sentiment_binary: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- char_count: integer (nullable = true)
 |-- word_count: integer (nullable = true)
 |-- avg_word_length: float (nullable = true)
 |-- exclamation_count: integer (nullable = true)
 |-- question_count: integer (nullable = true)
 |-- uppercase_count: integer (nullable = true)
 |-- uppercase_ratio: float (nullable = true)
 |-- sentence_count: integer (nullable = true)
 |-- useful: integer (nullable = false)
 |-- funny: integer (nullable = false)
 |-- cool: integer (nullable = false)
 |-- total_votes: integer (nullable = false)
 |-- date: timestamp (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)

In [40]:
# Save Gold layer
gold_final.write.mode('overwrite').parquet(f'{GOLD_DIR}/gold_reviews.parquet')
print(f'Gold layer saved → {GOLD_DIR}/gold_reviews.parquet')
print(f'Total features: {len(gold_final.columns)}')
print(f'Total rows    : {gold_count:,}')

Gold layer saved → /content/drive/MyDrive/Dataset/results/gold/gold_reviews.parquet
Total features: 45
Total rows    : 6,990,280


---
# Gold Layer EDA — Feature Analysis

## 22 — Sentiment Label Distribution

In [41]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable Distribution — Gold Layer', fontsize=14, fontweight='bold')

# 3-class
label_pd = gold_final.groupBy('sentiment_label').count().orderBy('sentiment_label').toPandas()
label_names = {0: 'Negative\n(1–2★)', 1: 'Neutral\n(3★)', 2: 'Positive\n(4–5★)'}
label_colors = ['#e74c3c', '#f39c12', '#2ecc71']
axes[0].bar([label_names[l] for l in label_pd['sentiment_label']],
            label_pd['count'], color=label_colors)
for bar, val in zip(axes[0].patches, label_pd['count']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10000,
                 f'{val/1e6:.2f}M', ha='center', fontsize=10)
axes[0].set_title('3-Class Sentiment Distribution')
axes[0].set_ylabel('Number of Reviews')

# Binary
bin_pd = gold_final.filter(F.col('sentiment_binary').isNotNull()) \
    .groupBy('sentiment_binary').count().orderBy('sentiment_binary').toPandas()
axes[1].pie(bin_pd['count'], labels=['Negative (1–2★)', 'Positive (4–5★)'],
            colors=['#e74c3c', '#2ecc71'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Binary Sentiment Distribution (excl. neutral)')

plt.tight_layout()
save_fig('10_gold_label_distribution')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/10_gold_label_distribution.png


## 23 — Feature Distributions by Sentiment

In [42]:
feat_sample = gold_final.select(
    'sentiment_label', 'char_count', 'word_count', 'exclamation_count',
    'uppercase_ratio', 'total_votes', 'user_review_count', 'biz_stars'
).sample(fraction=0.03, seed=42).toPandas()

feat_sample['sentiment'] = feat_sample['sentiment_label'].map(
    {0: 'Negative', 1: 'Neutral', 2: 'Positive'})

features_to_plot = [
    ('char_count',        'Character Count'),
    ('word_count',        'Word Count'),
    ('exclamation_count', 'Exclamation Count'),
    ('uppercase_ratio',   'Uppercase Ratio'),
    ('total_votes',       'Total Votes'),
    ('biz_stars',         'Business Avg Stars'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Feature Distributions by Sentiment Class', fontsize=15, fontweight='bold')
palette = {'Negative': '#e74c3c', 'Neutral': '#f39c12', 'Positive': '#2ecc71'}

for ax, (feat, title) in zip(axes.flatten(), features_to_plot):
    plot_data = feat_sample[feat_sample[feat] < feat_sample[feat].quantile(0.98)]
    sns.boxplot(data=plot_data, x='sentiment', y=feat,
                palette=palette, ax=ax, order=['Negative','Neutral','Positive'])
    ax.set_title(title); ax.set_xlabel('')

plt.tight_layout()
save_fig('11_gold_feature_distributions')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/11_gold_feature_distributions.png


## 24 — Correlation Heatmap (Numeric Features)

In [43]:
numeric_cols = [
    'stars', 'char_count', 'word_count', 'avg_word_length',
    'exclamation_count', 'question_count', 'uppercase_ratio',
    'sentence_count', 'useful', 'funny', 'cool', 'total_votes',
    'user_review_count', 'user_avg_stars', 'user_fans',
    'user_is_elite', 'user_tenure_days', 'user_engagement_score',
    'biz_stars', 'biz_review_count', 'biz_is_open', 'biz_category_count',
    'user_star_deviation', 'biz_star_deviation',
]

corr_sample = gold_final.select(numeric_cols).sample(fraction=0.05, seed=42).toPandas()
corr_sample = corr_sample.fillna(corr_sample.median(numeric_only=True))
corr_matrix = corr_sample.corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot_kws={'size': 7}, linewidths=0.5)
plt.title('Feature Correlation Heatmap — Gold Layer', fontsize=15, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
save_fig('12_gold_correlation_heatmap')

  Saved → /content/drive/MyDrive/Dataset/results/EDA/12_gold_correlation_heatmap.png


## 25 — Save Summary Statistics CSV

In [44]:
summary_rows = []

for name, df in silver_datasets.items():
    num_stats = df.select([c for c, t in df.dtypes if t in ('int','bigint','double','float')])
    desc = num_stats.describe().toPandas().set_index('summary').T
    desc.insert(0, 'dataset', name)
    summary_rows.append(desc)

summary_df = pd.concat(summary_rows)
summary_path = f'{RESULTS_DIR}/summary_statistics.csv'
summary_df.to_csv(summary_path)
print(f'Summary stats saved → {summary_path}')
display(summary_df.head(20))

Summary stats saved → /content/drive/MyDrive/Dataset/results/summary_statistics.csv


summary,dataset,count,mean,stddev,min,max
latitude,business,150346,36.67115006506401,5.872758914095136,27.555126,53.679195
longitude,business,150346,-89.35733948803325,14.918501665784994,-120.09514,-73.200455
stars,business,150346,3.5967235576603303,0.974420750920136,1.0,5.0
review_count,business,150346,44.86656113232144,121.12013570117027,5,7568
is_open,business,150346,0.7961502135075094,0.4028599390900677,0,1
category_count,business,150243,4.450070885166031,2.2314616835962107,1,36
stars,review,6990280,3.74858374771826,1.4787045052556862,1,5
useful,review,6990280,1.1846089140921394,3.253766966933352,-1,1182
funny,review,6990280,0.32655959417934616,1.6887290985540457,-1,792
cool,review,6990280,0.4986175088837643,2.1724598202111776,-1,404


---
## Results Summary

| Artifact | Location |
|----------|----------|
| EDA plots (12 PNG files) | `Dataset/results/EDA/` |
| Silver: business, review, user, tip, checkin | `Dataset/results/silver/` |
| Gold: `gold_reviews.parquet` | `Dataset/results/gold/` |
| Summary statistics CSV | `Dataset/results/summary_statistics.csv` |

### Gold Layer Features (ready for ML)
| Group | Features |
|-------|----------|
| **Text** | char_count, word_count, avg_word_length, exclamation_count, question_count, uppercase_ratio, sentence_count |
| **Temporal** | review_year, review_month, review_dow, review_hour, is_weekend |
| **User** | user_review_count, user_avg_stars, user_fans, user_is_elite, user_elite_years, user_tenure_days, user_engagement_score |
| **Business** | biz_stars, biz_review_count, biz_is_open, biz_category_count, biz_state |
| **Interaction** | user_star_deviation, biz_star_deviation, total_votes |
| **Labels** | sentiment_label (0/1/2), sentiment_binary (0/1) |